# Synthetic MRI Generation — Example

Demonstrates `dissector.creation.generate_synthetic`, which takes one real Dixon
water + fat-fraction scan pair and produces *n* augmented synthetic variants using
TorchIO (random flips, affine transforms, elastic deformation, bias field, noise,
and gamma contrast variation).

The example below generates **2 synthetic pairs** from `HV001_1_WATER_stack1`.

## Environment setup

Before running this notebook, create and activate a conda environment with the
required dependencies.

### 1 — Create the environment file

Save the following as `environment_creation.yml` (e.g. next to this notebook):

```yaml
name: mri_creation
channels:
  - conda-forge
  - defaults
dependencies:
  - python=3.10
  - pip
  - pip:
      - torchio
      - SimpleITK
      - matplotlib
      - numpy
      - jupyter
```

### 2 — Create and activate the environment

```bash
conda env create -f environment_creation.yml
conda activate mri_creation
```

### 3 — Launch Jupyter and open this notebook

```bash
jupyter notebook synthetic_mri_example.ipynb
```

### 4 — To update the environment later (e.g. add a package)

Add the package under `pip:` in the yml file, then:

```bash
conda env update -f environment_creation.yml --prune
```

### Warning: environment. This notebook works in a different environment. Essentially you need torchio, sitk,  matplotlib and numpy in your environment. 

In [ ]:
import sys, pathlib
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.pyplot as plt
import SimpleITK as sitk
sys.path.insert(0, str(pathlib.Path('..', 'src').resolve()))

from dissector.creation import generate_augmented
from dissector.creation import register_and_blend

In [ ]:
EVAL_DIR = pathlib.Path('.')

water_path = EVAL_DIR / 'myosegmenTUM' / 'HV001_1' / 'ImageData' / 'HV001_1_WATER' / 'HV001_1_WATER_stack1.nii'
fat_path   = EVAL_DIR / 'myosegmenTUM' / 'HV001_1' / 'ImageData' / 'HV001_1_FATFRACTION' / 'HV001_1_FATFRACTION_stack1.nii'
output_dir = EVAL_DIR / 'synthetic_output'

print('Water exists:', water_path.exists())
print('Fat exists  :', fat_path.exists())

In [ ]:
pairs = generate_augmented(
    water_path=water_path,
    fat_path=fat_path,
    n=2,
    output_dir=output_dir,
)

print(f'\nGenerated {len(pairs)} augmented "synthetic" pair(s):')
for w, f in pairs:
    print(f'  water : {w}')
    print(f'  fat   : {f}')

In [ ]:
# Quick visual check — compare one slice of the original vs both augmented


def mid_slice(path):
    arr = sitk.GetArrayFromImage(sitk.ReadImage(str(path))).astype(float)
    s = arr[arr.shape[0] // 2]
    return (s - s.min()) / (s.max() - s.min() + 1e-8)

titles = ['Original water', 'Synth 000 water', 'Synth 001 water']
images = [water_path, pairs[0][0], pairs[1][0]]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, title, path in zip(axes, titles, images):
    ax.imshow(mid_slice(path), cmap='gray', origin='lower')
    ax.set_title(title)
    ax.axis('off')

fig.suptitle('Middle slice — original vs synthetic water images', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Register augmented image 1 onto augmented image 0, then blend them


blended_water = register_and_blend(
    fixed_path=pairs[0][0],
    moving_path=pairs[1][0],
    output_path=output_dir / 'blended_water.nii.gz',
    alpha=0.5,
    transform='affine',
)

blended_fat = register_and_blend(
    fixed_path=pairs[0][1],
    moving_path=pairs[1][1],
    output_path=output_dir / 'blended_fat.nii.gz',
    alpha=0.5,
    transform='affine',
)

print('Registration and blending complete.')

In [ ]:
# Visual check — augmented 0, augmented 1, and their blend


titles = ['Augmented 0', 'Augmented 1', 'Blended (50/50)']
images = [pairs[0][0], pairs[1][0], output_dir / 'blended_water.nii.gz']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, title, path in zip(axes, titles, images):
    ax.imshow(mid_slice(path), cmap='gray', origin='lower')
    ax.set_title(title)
    ax.axis('off')

fig.suptitle('Middle slice — two augmented images and their registered blend', fontsize=12)
plt.tight_layout()
plt.show()

## Truly synthetic images from segmentation mask

`generate_synthetic` takes a ground-truth segmentation mask instead of a real image. This is only experimental, as we don't have ground truth masks with all the muscles!
Each tissue label is assigned random Gaussian intensity statistics, smoothed at
label boundaries to simulate partial-volume effects, then corrupted with a random
bias field.  No real pixel values are reused — every output looks like a different scanner/subject.

In [ ]:
from dissector.creation import generate_synthetic

seg_path = EVAL_DIR / 'myosegmenTUM' / 'HV001_1' / 'SegmentationMasks' / 'combined_gt_stack1.mha'

synth_pairs = generate_synthetic(
    seg_path=seg_path,
    n=2,
    output_dir=output_dir / 'truly_synthetic',
    seed=42,
)

print(f'\nGenerated {len(synth_pairs)} truly synthetic pair(s):')
for w, f in synth_pairs:
    print(f'  water : {w}')
    print(f'  fat   : {f}')

In [ ]:
# Visual comparison — original vs both truly synthetic water images
titles = ['Original water', 'Truly synth 000', 'Truly synth 001']
images = [water_path, synth_pairs[0][0], synth_pairs[1][0]]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, title, path in zip(axes, titles, images):
    ax.imshow(mid_slice(path), cmap='gray', origin='lower')
    ax.set_title(title)
    ax.axis('off')

fig.suptitle('Middle slice — original vs truly synthetic (from segmentation mask)', fontsize=12)
plt.tight_layout()
plt.show()